# Bước B3 (Hướng A — Dataset lớn 5 phiên): `DINOv3` + `NIW` + `FTR1` + `DYDM` trên `CUB-200`
- **Nhánh:** `duytung/11-cub-dinov3-dydm`
- **Dataset:** `CUB-200` (`11.788` ảnh, `200` lớp chim hạt mịn, **5 phiên liên tiếp `S1..S5`** — tự động tải từ HuggingFace `cassiekang/cub200_dataset`, **không cần upload Drive!**).
- **Mốc đối chiếu:** Bài báo gốc `VB-CGCD (ICML 2025)` (`S5_All = 66.44%`, `M_d = 56.46%`) | Đỉnh Nhóm `FTR1 DINOv1` (`74.68%`) | Bước B2 `DYDM_km DINOv1` (`75.02%`).
- **Thời gian dự kiến (GPU L4):** ~80–90 phút trọn bộ (Phase 1 + 5 seed Phase 2 + 5 seed DYDM), hỗ trợ **auto-resume từng bước trên Google Drive**.

In [ ]:
# Cell 1: Kết nối Google Drive & kiểm tra GPU
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
# Cell 2: Clone/pull đúng nhánh duytung/11-cub-dinov3-dydm
import os, subprocess

REPO_URL = 'https://github.com/DuytungDeveloper/VB-CGCD-defect-testing-only.git'
BRANCH = 'duytung/11-cub-dinov3-dydm'
REPO_DIR = '/content/VB-CGCD-defect-testing-only'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'], check=True)

subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--oneline'], check=True)

In [ ]:
# Cell 3: Cài đặt thư viện (nếu runtime mới)
!pip install -q peft transformers accelerate scipy scikit-learn datasets psutil prettytable
!python -c "import torch, jax, transformers, peft; print('Torch:', torch.__version__, 'CUDA:', torch.cuda.is_available(), '| JAX:', jax.__version__, jax.devices())"

In [ ]:
# Cell 4: Chạy nền Bước B3 (CUB-200 DINOv3 + DYDM) có auto-resume trên Google Drive
import os, subprocess
from google.colab import userdata

try:
    hf_tok = userdata.get('HF_TOKEN')
except Exception:
    hf_tok = os.environ.get('HF_TOKEN', '')

assert hf_tok, 'Vui lòng cấu hình HF_TOKEN trong Colab Secrets (biểu tượng chìa khóa bên trái)!'

env = os.environ.copy()
env['HF_TOKEN'] = hf_tok
env['HUGGING_FACE_HUB_TOKEN'] = hf_tok

log_path = '/content/drive/MyDrive/VB_CGCD_Work/cub_b3_dinov3_dydm.log'
os.makedirs(os.path.dirname(log_path), exist_ok=True)
pid_path = '/tmp/cub_b3.pid'

with open(log_path, 'a') as lf:
    p = subprocess.Popen(
        ['bash', '/content/VB-CGCD-defect-testing-only/colab/run_cub_b3_bg.sh'],
        stdout=lf, stderr=subprocess.STDOUT, env=env, start_new_session=True
    )
with open(pid_path, 'w') as pf:
    pf.write(str(p.pid))
print(f'Đã khởi chạy Bước B3 (CUB-200 DINOv3 + DYDM) chạy nền! PID={p.pid} | Log: {log_path}')

In [ ]:
# Cell 5: Kiểm tra tiến độ & xem bảng tổng hợp khi chạy xong
from pathlib import Path

pid = int(Path('/tmp/cub_b3.pid').read_text().strip()) if Path('/tmp/cub_b3.pid').exists() else -1
alive = False
if pid > 0 and Path(f'/proc/{pid}/status').exists():
    st = Path(f'/proc/{pid}/status').read_text()
    alive = ('State:\tZ' not in st and 'State: Z' not in st)
print(f'Trạng thái PID={pid}:', 'ĐANG CHẠY' if alive else 'ĐÃ DỪNG / XONG')

log_p = Path('/content/drive/MyDrive/VB_CGCD_Work/cub_b3_dinov3_dydm.log')
if log_p.exists():
    lines = log_p.read_text(errors='replace').splitlines()
    print('\n--- 40 dòng log gần nhất ---')
    print('\n'.join(lines[-40:]))

sum_p = Path('/content/drive/MyDrive/VB_CGCD_Work/cub_b3_dinov3_dydm_summary.txt')
if sum_p.exists() and not alive:
    print('\n--- BẢNG TỔNG HỢP BƯỚC B3 (CUB-200 DINOv3 + DYDM) ---')
    print(sum_p.read_text())